# DeepSeek-Math-7B SFT/RL: CommonsenseQA pass@8 only (200 examples, 4-bit)

## Notes on this version

- Runs **CommonsenseQA pass@8 only**.
- Default limit: **first 200 CommonsenseQA validation examples**.
- IFEval is disabled and not loaded.
- Uses **4-bit NF4 quantization** to avoid the previous GPU memory / CPU offload error.
- Keep `MODEL_KEYS_TO_RUN = ["sft"]` for SFT only; change to `["sft", "rl"]` only if you want both official models.

In [1]:
!pip install -q --upgrade transformers accelerate datasets evaluate sentencepiece tqdm protobuf safetensors
!pip install -q --upgrade bitsandbytes
!pip install -q absl-py immutabledict nltk langdetect

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 148.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 49.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 10.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 324.3/324.3 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 34.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opentelemetry-proto 1.38.0 requires protobuf<7.0,>=5.0, but you have protobuf 7.34.1 which is incompatible.
google-ai-generativelanguage 0.6.15 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<6.0.0dev,>=3.20.2, but you have protobuf 7.34.1 which is incompatible.
google-cloud-aiplatform 1.148.1 requires protobuf!=4.21.0,!=4.21.1,!=4.21.2,!=4.21.3,!=4.21.4,!=4.21.5,<7.0.0,>=3.20.2, but you have proto

In [2]:
from pathlib import Path
import os
import gc
import re
import json
import shutil
import subprocess
from typing import Any, Dict, Optional, List, Tuple

import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

# -----------------------------
# Models
# -----------------------------
MODEL_CONFIGS = {
    "sft": {
        "display_name": "DeepSeek-Math-7B-Instruct",
        "model_name": "deepseek-ai/deepseek-math-7b-instruct",
        "stage": "SFT / Instruct",
        "output_dir": "deepseek_math_7b_instruct_results",
    },
    "rl": {
        "display_name": "DeepSeek-Math-7B-RL",
        "model_name": "deepseek-ai/deepseek-math-7b-rl",
        "stage": "RL / GRPO",
        "output_dir": "deepseek_math_7b_rl_results",
    },
}

# -----------------------------
# What to run
# -----------------------------
# Start with SFT. Change to ["sft", "rl"] when you want to run both official models.
MODEL_KEYS_TO_RUN = ["sft"]
RUN_CSQA_PASS8 = True
RUN_IFEVAL_PASS8 = False
RUN_IFEVAL_OFFICIAL = False

# Optional: also generate greedy pass@1 in the same run.
# If you already ran pass@1 before, keep this False to save time.
RUN_GREEDY_PASS1 = False

# -----------------------------
# Evaluation limits
# -----------------------------
# Use None for full validation/train set.
CSQA_LIMIT = 200           # Run first 200 CommonsenseQA validation examples.
IFEVAL_LIMIT = None          # IFEval is disabled in this notebook.

# For quick testing, uncomment:
# CSQA_LIMIT = 200           # Run first 200 CommonsenseQA validation examples.
# IFEVAL_LIMIT = None          # IFEval is disabled in this notebook.

# -----------------------------
# pass@8 generation settings
# -----------------------------
NUM_SAMPLES = 8
PASS8_TEMPERATURE = 0.7
PASS8_TOP_P = 0.95

CSQA_MAX_NEW_TOKENS = 32
IFEVAL_MAX_NEW_TOKENS = 512

# Print progress every N examples instead of using tqdm.
PROGRESS_EVERY = 25

# -----------------------------
# Loading settings
# -----------------------------
# Use 4-bit NF4 quantization to reduce GPU memory usage.
# This fixes the 8-bit device_map CPU/disk offload error on small Colab GPUs.
USE_4BIT = True
USE_8BIT = False
USE_FLOAT16 = True         # Used only if USE_4BIT = False and USE_8BIT = False.
TRUST_REMOTE_CODE = True

# DeepSeek-Math instruction format. Use the same wrapper for SFT and RL.
USE_DEEPSEEK_USER_ASSISTANT_WRAPPER = True

BASE_OUTPUT_DIR = Path("deepseek_math_eval_results")
BASE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Optional reproducibility. We set the seed once instead of resetting before every sample.
# This keeps samples diverse while making runs easier to reproduce.
RANDOM_SEED = 1234
torch.manual_seed(RANDOM_SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(RANDOM_SEED)


print("Will run models:", MODEL_KEYS_TO_RUN)
print("RUN_CSQA_PASS8:", RUN_CSQA_PASS8)
print("RUN_IFEVAL_PASS8:", RUN_IFEVAL_PASS8)
print("RUN_GREEDY_PASS1:", RUN_GREEDY_PASS1)
print("NUM_SAMPLES:", NUM_SAMPLES)
print("Output base dir:", BASE_OUTPUT_DIR.resolve())

Will run models: ['sft']
RUN_CSQA_PASS8: True
RUN_IFEVAL_PASS8: False
RUN_GREEDY_PASS1: False
NUM_SAMPLES: 8
Output base dir: /content/deepseek_math_eval_results


In [3]:
def load_deepseek_model(model_name: str):
    """Load tokenizer and model with memory-safe 4-bit quantization by default."""
    print(f"Loading {model_name} ...")

    # Clean possible leftover GPU memory before loading.
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

    tokenizer = AutoTokenizer.from_pretrained(
        model_name,
        trust_remote_code=TRUST_REMOTE_CODE,
    )

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token = tokenizer.eos_token

    if USE_4BIT:
        bnb_config = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.float16,
            bnb_4bit_quant_type="nf4",
            bnb_4bit_use_double_quant=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            quantization_config=bnb_config,
            trust_remote_code=TRUST_REMOTE_CODE,
            low_cpu_mem_usage=True,
        )
    elif USE_8BIT:
        # If you switch back to 8-bit and the GPU is too small, CPU offload is enabled.
        bnb_config = BitsAndBytesConfig(
            load_in_8bit=True,
            llm_int8_enable_fp32_cpu_offload=True,
        )
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            quantization_config=bnb_config,
            trust_remote_code=TRUST_REMOTE_CODE,
            torch_dtype=torch.float16,
            low_cpu_mem_usage=True,
        )
    else:
        dtype = torch.float16 if USE_FLOAT16 and torch.cuda.is_available() else torch.float32
        model = AutoModelForCausalLM.from_pretrained(
            model_name,
            device_map="auto",
            torch_dtype=dtype,
            trust_remote_code=TRUST_REMOTE_CODE,
            low_cpu_mem_usage=True,
        )

    model.eval()
    print("Loaded:", model_name)
    return tokenizer, model

def format_for_deepseek(task_prompt: str) -> str:
    """Wrap task prompt for DeepSeek-Math Instruct/RL models."""
    if USE_DEEPSEEK_USER_ASSISTANT_WRAPPER:
        return f"User: {task_prompt}\nAssistant:"
    return task_prompt


def get_first_device(model):
    return next(model.parameters()).device


def clean_generation_text(text: str) -> str:
    """Remove common tokenizer artifacts seen in DeepSeek decoded outputs."""
    if text is None:
        return ""
    text = text.replace("Ġ", " ").replace("Ċ", "\n").replace("ĉ", "\t")
    return text.strip()


def generate_text(
    tokenizer,
    model,
    task_prompt: str,
    max_new_tokens: int = 128,
    temperature: float = 0.0,
    top_p: Optional[float] = None,
    sample_id: Optional[int] = None,
) -> str:
    """Generate text from a task prompt and return only newly generated tokens."""
    prompt = format_for_deepseek(task_prompt)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        return_token_type_ids=False,
    )

    first_device = get_first_device(model)
    inputs = {k: v.to(first_device) for k, v in inputs.items()}

    do_sample = temperature > 0

    gen_kwargs = {
        "max_new_tokens": max_new_tokens,
        "do_sample": do_sample,
        "pad_token_id": tokenizer.pad_token_id,
        "eos_token_id": tokenizer.eos_token_id,
    }

    if do_sample:
        gen_kwargs["temperature"] = temperature
        if top_p is not None:
            gen_kwargs["top_p"] = top_p

    # Do not reset the RNG inside each sample.
    # Resetting here can reduce diversity and makes checkpoint-resumed runs less natural.

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            **gen_kwargs,
        )

    input_len = inputs["input_ids"].shape[1]
    new_tokens = outputs[0][input_len:]
    decoded = tokenizer.decode(new_tokens, skip_special_tokens=True)
    return clean_generation_text(decoded)


def generate_n_samples(
    tokenizer,
    model,
    task_prompt: str,
    max_new_tokens: int,
    n: int = NUM_SAMPLES,
    temperature: float = PASS8_TEMPERATURE,
    top_p: float = PASS8_TOP_P,
) -> List[str]:
    outputs = []
    for sample_idx in range(n):
        outputs.append(
            generate_text(
                tokenizer=tokenizer,
                model=model,
                task_prompt=task_prompt,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
                top_p=top_p,
                sample_id=sample_idx,
            )
        )
    return outputs


def unload_model(tokenizer=None, model=None):
    """Free GPU memory between models."""
    try:
        del model
    except Exception:
        pass
    try:
        del tokenizer
    except Exception:
        pass
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
    print("Freed memory.")

In [4]:
# Load datasets
csqa_full = load_dataset("tau/commonsense_qa", split="validation")
if CSQA_LIMIT is not None:
    csqa = csqa_full.select(range(min(CSQA_LIMIT, len(csqa_full))))
else:
    csqa = csqa_full

print("CommonsenseQA:", csqa)
print("CSQA example:", csqa[0])

# IFEval is disabled in this notebook. Do not load it unless RUN_IFEVAL_PASS8 = True.
ifeval = None
if RUN_IFEVAL_PASS8:
    ifeval_full = load_dataset("google/IFEval", split="train")
    if IFEVAL_LIMIT is not None:
        ifeval = ifeval_full.select(range(min(IFEVAL_LIMIT, len(ifeval_full))))
    else:
        ifeval = ifeval_full
    print("IFEval:", ifeval)
    print("IFEval example:", ifeval[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/1.25M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/160k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/151k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/9741 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/1221 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/1140 [00:00<?, ? examples/s]

CommonsenseQA: Dataset({
    features: ['id', 'question', 'question_concept', 'choices', 'answerKey'],
    num_rows: 200
})
CSQA example: {'id': '1afa02df02c908a558b4036e80242fac', 'question': 'A revolving door is convenient for two direction travel, but it also serves as a security measure at a what?', 'question_concept': 'revolving door', 'choices': {'label': ['A', 'B', 'C', 'D', 'E'], 'text': ['bank', 'library', 'department store', 'mall', 'new york']}, 'answerKey': 'A'}


In [5]:
def build_csqa_prompt(ex: Dict[str, Any]) -> str:
    labels = ex["choices"]["label"]
    texts = ex["choices"]["text"]
    options = "\n".join([f"{label}. {text}" for label, text in zip(labels, texts)])

    return (
        "Answer the following multiple-choice question.\n"
        "Choose the best answer from A, B, C, D, and E.\n"
        "Return only one letter: A, B, C, D, or E.\n\n"
        f"Question: {ex['question']}\n"
        f"{options}\n\n"
        "Answer:"
    )


def extract_csqa_answer(text: str) -> Optional[str]:
    """Extract A/B/C/D/E robustly from DeepSeek outputs."""
    if text is None:
        return None

    t = clean_generation_text(str(text))
    t_upper = t.upper()

    # 1. Catch math style: \boxed{A}
    match = re.search(r"\\BOXED\{\s*([ABCDE])\s*\}", t_upper)
    if match:
        return match.group(1)

    # 2. Catch "Answer: A" / "The answer is A" / "Answer is: A"
    match = re.search(r"ANSWER\s*(?:IS)?\s*[:\-]?\s*([ABCDE])\b", t_upper)
    if match:
        return match.group(1)

    # 3. Catch output starting with A, A., A), A: etc.
    match = re.match(r"^\s*([ABCDE])\s*(?:[\.\)\:\-]|\b)", t_upper)
    if match:
        return match.group(1)

    # 4. Conservative fallback: standalone letter only.
    match = re.search(r"\b([ABCDE])\b", t_upper)
    if match:
        return match.group(1)

    return None


def evaluate_commonsenseqa_pass8(tokenizer, model, model_info: Dict[str, str], output_dir: Path) -> Dict[str, Any]:
    """Empirical pass@8 on CommonsenseQA: correct if any sampled answer matches gold."""
    predictions = []
    pass1_correct = 0
    pass8_correct = 0
    pass1_parsed = 0
    any_parsed = 0

    total = len(csqa)
    print(f"CommonsenseQA pass@8: {model_info['display_name']} | total={total} | samples={NUM_SAMPLES}")

    for i, ex in enumerate(csqa, start=1):
        if i == 1 or i % PROGRESS_EVERY == 0 or i == total:
            print(f"CommonsenseQA pass@8 progress: {i}/{total}", flush=True)

        prompt = build_csqa_prompt(ex)
        gold = ex["answerKey"]

        sampled_outputs = generate_n_samples(
            tokenizer=tokenizer,
            model=model,
            task_prompt=prompt,
            max_new_tokens=CSQA_MAX_NEW_TOKENS,
            n=NUM_SAMPLES,
            temperature=PASS8_TEMPERATURE,
            top_p=PASS8_TOP_P,
        )
        sampled_preds = [extract_csqa_answer(x) for x in sampled_outputs]

        # pass@1 here is sample-0 pass@1 under the same sampling setting.
        # If you need deterministic greedy pass@1, set RUN_GREEDY_PASS1=True.
        pass1_pred = sampled_preds[0] if sampled_preds else None
        if pass1_pred is not None:
            pass1_parsed += 1
        if any(p is not None for p in sampled_preds):
            any_parsed += 1
        if pass1_pred == gold:
            pass1_correct += 1
        if any(p == gold for p in sampled_preds):
            pass8_correct += 1

        item = {
            "id": ex["id"],
            "question": ex["question"],
            "gold": gold,
            "sampled_predictions": sampled_preds,
            "sampled_raw_outputs": sampled_outputs,
            "pass1_sample0_correct": pass1_pred == gold,
            "pass8_correct": any(p == gold for p in sampled_preds),
        }

        if RUN_GREEDY_PASS1:
            greedy_output = generate_text(
                tokenizer=tokenizer,
                model=model,
                task_prompt=prompt,
                max_new_tokens=CSQA_MAX_NEW_TOKENS,
                temperature=0.0,
            )
            greedy_pred = extract_csqa_answer(greedy_output)
            item["greedy_prediction"] = greedy_pred
            item["greedy_raw_output"] = greedy_output
            item["greedy_correct"] = greedy_pred == gold

        predictions.append(item)

    metrics = {
        "model": model_info["model_name"],
        "display_name": model_info["display_name"],
        "stage": model_info["stage"],
        "benchmark": "CommonsenseQA",
        "split": "validation",
        "num_examples": len(csqa),
        "num_samples": NUM_SAMPLES,
        "pass_at_1_sample0_accuracy": pass1_correct / len(csqa),
        "pass_at_8_accuracy": pass8_correct / len(csqa),
        "pass_at_1_sample0_parse_rate": pass1_parsed / len(csqa),
        "any_sample_parse_rate": any_parsed / len(csqa),
        "sampling_temperature": PASS8_TEMPERATURE,
        "sampling_top_p": PASS8_TOP_P,
    }

    if RUN_GREEDY_PASS1:
        greedy_correct = sum(1 for x in predictions if x.get("greedy_correct") is True)
        greedy_parsed = sum(1 for x in predictions if x.get("greedy_prediction") is not None)
        metrics["greedy_pass_at_1_accuracy"] = greedy_correct / len(csqa)
        metrics["greedy_parse_rate"] = greedy_parsed / len(csqa)

    output_dir.mkdir(parents=True, exist_ok=True)
    suffix = f"{len(csqa)}_n{NUM_SAMPLES}"
    pred_file = output_dir / f"commonsenseqa_pass8_predictions_{suffix}.json"
    metrics_file = output_dir / f"commonsenseqa_pass8_metrics_{suffix}.json"

    with open(pred_file, "w", encoding="utf-8") as f:
        json.dump(predictions, f, ensure_ascii=False, indent=2)

    with open(metrics_file, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    print(json.dumps(metrics, indent=2))
    print("Saved:", pred_file)
    print("Saved:", metrics_file)

    for item in predictions[:3]:
        print("=" * 80)
        print("Gold:", item["gold"])
        print("Sample preds:", item["sampled_predictions"])
        print("pass@8 correct:", item["pass8_correct"])
        print("Sample 0 raw:", item["sampled_raw_outputs"][0][:500])

    return metrics

In [6]:
def generate_ifeval_pass8(tokenizer, model, model_info: Dict[str, str], output_dir: Path) -> Path:
    """Generate 8 sampled responses per IFEval prompt with JSONL checkpointing.

    Each JSONL row stores one prompt with a list of `sampled_responses`.
    If Colab disconnects, rerun this cell; completed prompts are skipped.
    """
    output_dir.mkdir(parents=True, exist_ok=True)

    suffix = f"{len(ifeval)}_n{NUM_SAMPLES}"
    jsonl_file = output_dir / f"ifeval_pass8_generations_{suffix}.jsonl"
    gen_file = output_dir / f"ifeval_pass8_generations_{suffix}.json"

    done_indices = set()
    generations = []

    if jsonl_file.exists():
        with open(jsonl_file, "r", encoding="utf-8") as f:
            for line in f:
                if not line.strip():
                    continue
                item = json.loads(line)
                generations.append(item)
                done_indices.add(item["original_index"])
        print(f"Found existing partial IFEval pass@8 results: {len(done_indices)} examples")

    total = len(ifeval)
    print(
        f"IFEval pass@8 generation: {model_info['display_name']} | "
        f"total={total} | already_done={len(done_indices)} | samples={NUM_SAMPLES}"
    )

    with open(jsonl_file, "a", encoding="utf-8") as fout:
        for idx, ex in enumerate(ifeval):
            if idx in done_indices:
                continue

            current = idx + 1
            if current == 1 or current % PROGRESS_EVERY == 0 or current == total:
                print(f"IFEval pass@8 progress: {current}/{total}", flush=True)

            prompt = ex["prompt"]
            sampled_responses = generate_n_samples(
                tokenizer=tokenizer,
                model=model,
                task_prompt=prompt,
                max_new_tokens=IFEVAL_MAX_NEW_TOKENS,
                n=NUM_SAMPLES,
                temperature=PASS8_TEMPERATURE,
                top_p=PASS8_TOP_P,
            )

            item = {
                "original_index": idx,
                "key": ex.get("key", idx),
                "prompt": prompt,
                "instruction_id_list": ex["instruction_id_list"],
                "kwargs": ex["kwargs"],
                "sampled_responses": sampled_responses,
            }

            if RUN_GREEDY_PASS1:
                item["greedy_response"] = generate_text(
                    tokenizer=tokenizer,
                    model=model,
                    task_prompt=prompt,
                    max_new_tokens=IFEVAL_MAX_NEW_TOKENS,
                    temperature=0.0,
                )

            fout.write(json.dumps(item, ensure_ascii=False) + "\n")
            fout.flush()
            generations.append(item)
            done_indices.add(idx)

    generations = sorted(generations, key=lambda x: x["original_index"])

    with open(gen_file, "w", encoding="utf-8") as f:
        json.dump(generations, f, ensure_ascii=False, indent=2)

    print("Saved checkpoint JSONL:", jsonl_file)
    print("Saved evaluator JSON:", gen_file)
    print("Num examples:", len(generations))
    if generations:
        print("Example sampled response 0:")
        print(generations[0]["sampled_responses"][0][:1000])

    return gen_file

In [7]:
# Clone official IFEval evaluator only once.
if not Path("google-research").exists():
    !git clone --depth 1 https://github.com/google-research/google-research.git
else:
    print("google-research already exists.")

# NLTK data required by official IFEval evaluator.
import nltk
nltk.download("punkt", quiet=True)
nltk.download("punkt_tab", quiet=True)
print("NLTK punkt + punkt_tab ready.")

Cloning into 'google-research'...
remote: Enumerating objects: 23308, done.
remote: Counting objects: 100% (23308/23308), done.
remote: Compressing objects: 100% (18089/18089), done.
remote: Total 23308 (delta 4694), reused 14612 (delta 4032), pack-reused 0 (from 0)
Receiving objects: 100% (23308/23308), 482.87 MiB | 1.87 MiB/s, done.
Resolving deltas: 100% (4694/4694), done.
Updating files: 100% (22171/22171), done.
NLTK punkt + punkt_tab ready.


In [8]:
def clean_dict(d):
    """Remove keys whose values are None."""
    if d is None:
        return {}
    return {k: v for k, v in d.items() if v is not None}


def normalize_ifeval_kwargs(kwargs, instruction_id_list):
    """Convert HF google/IFEval kwargs to official evaluator format."""
    n = len(instruction_id_list)

    if kwargs is None:
        return [{} for _ in range(n)]

    if isinstance(kwargs, list):
        fixed = []
        for item in kwargs:
            if isinstance(item, dict):
                fixed.append(clean_dict(item))
            else:
                fixed.append({})
        return fixed

    if isinstance(kwargs, dict):
        fixed = [{} for _ in range(n)]
        for key, value in kwargs.items():
            if isinstance(value, list):
                for i in range(min(n, len(value))):
                    if value[i] is not None:
                        fixed[i][key] = value[i]
            else:
                if value is not None:
                    for i in range(n):
                        fixed[i][key] = value
        return [clean_dict(x) for x in fixed]

    return [{} for _ in range(n)]


def load_ifeval_pass8_generations(generation_file: Path) -> List[Dict[str, Any]]:
    with open(generation_file, "r", encoding="utf-8") as f:
        return json.load(f)


def write_ifeval_input_file(generations: List[Dict[str, Any]], output_dir: Path) -> Path:
    input_path = output_dir / "ifeval_pass8_input.jsonl"

    with open(input_path, "w", encoding="utf-8") as fin:
        for i, ex in enumerate(generations):
            key = ex.get("key", None)
            if key is None:
                key = ex.get("original_index", i)

            instruction_id_list = ex["instruction_id_list"]
            fixed_kwargs = normalize_ifeval_kwargs(ex["kwargs"], instruction_id_list)

            input_obj = {
                "key": key,
                "instruction_id_list": instruction_id_list,
                "prompt": ex["prompt"],
                "kwargs": fixed_kwargs,
            }
            fin.write(json.dumps(input_obj, ensure_ascii=False) + "\n")

    print("Wrote:", input_path)
    return input_path


def write_ifeval_response_file(
    generations: List[Dict[str, Any]],
    output_dir: Path,
    sample_idx: Optional[int] = None,
    use_greedy: bool = False,
) -> Path:
    if use_greedy:
        response_path = output_dir / "ifeval_greedy_response.jsonl"
    else:
        response_path = output_dir / f"ifeval_pass8_response_sample_{sample_idx}.jsonl"

    with open(response_path, "w", encoding="utf-8") as fout:
        for i, ex in enumerate(generations):
            key = ex.get("key", None)
            if key is None:
                key = ex.get("original_index", i)

            if use_greedy:
                response = ex["greedy_response"]
            else:
                response = ex["sampled_responses"][sample_idx]

            response_obj = {
                "key": key,
                "prompt": ex["prompt"],
                "response": response,
            }
            fout.write(json.dumps(response_obj, ensure_ascii=False) + "\n")

    print("Wrote:", response_path)
    return response_path


def run_single_ifeval_official(input_path: Path, response_path: Path, official_dir: Path) -> Path:
    if official_dir.exists():
        shutil.rmtree(official_dir)
    official_dir.mkdir(parents=True, exist_ok=True)

    cmd = [
        "python",
        "google-research/instruction_following_eval/evaluation_main.py",
        f"--input_data={input_path}",
        f"--input_response_data={response_path}",
        f"--output_dir={official_dir}",
    ]

    print("Running:", " ".join(cmd))

    env = os.environ.copy()
    env["PYTHONPATH"] = "google-research"

    result = subprocess.run(cmd, env=env, text=True, capture_output=True)

    if result.stdout:
        print("STDOUT:")
        print(result.stdout)
    if result.stderr:
        print("STDERR:")
        print(result.stderr)

    if result.returncode != 0:
        raise RuntimeError(f"Official IFEval evaluator failed with return code {result.returncode}")

    return official_dir / "eval_results_strict.jsonl"


def read_ifeval_records(strict_file: Path) -> List[Dict[str, Any]]:
    records = []
    with open(strict_file, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records


def metrics_from_ifeval_records(records: List[Dict[str, Any]]) -> Dict[str, float]:
    prompt_correct = 0
    instruction_correct = 0
    instruction_total = 0

    for r in records:
        follow_instruction_list = r["follow_instruction_list"]
        if all(follow_instruction_list):
            prompt_correct += 1
        instruction_correct += sum(follow_instruction_list)
        instruction_total += len(follow_instruction_list)

    return {
        "num_examples": len(records),
        "strict_prompt_accuracy": prompt_correct / len(records),
        "strict_instruction_accuracy": instruction_correct / instruction_total,
    }


def combine_ifeval_pass8(sample_records: List[List[Dict[str, Any]]]) -> Dict[str, float]:
    """Combine official IFEval strict outputs into empirical pass@8 metrics.

    prompt-level pass@8: at least one sample satisfies all instructions for a prompt.
    instruction-level pass@8: for each instruction, at least one sample satisfies that instruction.

    This version aligns records by key instead of relying only on file order.
    """
    assert len(sample_records) == NUM_SAMPLES

    keyed_records = []
    for records in sample_records:
        keyed_records.append({str(r.get("key", i)): r for i, r in enumerate(records)})

    keys = list(keyed_records[0].keys())
    for kr in keyed_records[1:]:
        assert set(kr.keys()) == set(keys), "IFEval sample outputs have mismatched keys"

    prompt_pass8_correct = 0
    instruction_pass8_correct = 0
    instruction_total = 0

    for key in keys:
        per_sample_lists = [kr[key]["follow_instruction_list"] for kr in keyed_records]
        num_instr = len(per_sample_lists[0])
        assert all(len(lst) == num_instr for lst in per_sample_lists), "Mismatched instruction counts"

        if any(all(lst) for lst in per_sample_lists):
            prompt_pass8_correct += 1

        for j in range(num_instr):
            instruction_ok = any(lst[j] for lst in per_sample_lists)
            instruction_pass8_correct += int(instruction_ok)
            instruction_total += 1

    return {
        "num_examples": len(keys),
        "num_samples": NUM_SAMPLES,
        "strict_prompt_pass_at_8": prompt_pass8_correct / len(keys),
        "strict_instruction_pass_at_8": instruction_correct_safe(instruction_pass8_correct, instruction_total),
    }


def instruction_correct_safe(correct: int, total: int) -> float:
    return correct / total if total else 0.0


def run_ifeval_official_pass8(generation_file: Path, output_dir: Path) -> Dict[str, Any]:
    generations = load_ifeval_pass8_generations(generation_file)
    input_path = write_ifeval_input_file(generations, output_dir)

    root_official_dir = output_dir / f"ifeval_official_pass8_{len(generations)}_n{NUM_SAMPLES}"
    root_official_dir.mkdir(parents=True, exist_ok=True)

    sample_metrics = []
    sample_records = []

    for sample_idx in range(NUM_SAMPLES):
        print("=" * 80)
        print(f"Running official IFEval evaluator for sample {sample_idx + 1}/{NUM_SAMPLES}")
        response_path = write_ifeval_response_file(
            generations=generations,
            output_dir=output_dir,
            sample_idx=sample_idx,
            use_greedy=False,
        )
        strict_file = run_single_ifeval_official(
            input_path=input_path,
            response_path=response_path,
            official_dir=root_official_dir / f"sample_{sample_idx}",
        )
        records = read_ifeval_records(strict_file)
        sample_records.append(records)
        sample_metrics.append(metrics_from_ifeval_records(records))

    pass8_metrics = combine_ifeval_pass8(sample_records)

    metrics = {
        "benchmark": "IFEval",
        "split": "train",
        "num_examples": len(generations),
        "num_samples": NUM_SAMPLES,
        "sampling_temperature": PASS8_TEMPERATURE,
        "sampling_top_p": PASS8_TOP_P,
        "sample0_strict_prompt_accuracy": sample_metrics[0]["strict_prompt_accuracy"],
        "sample0_strict_instruction_accuracy": sample_metrics[0]["strict_instruction_accuracy"],
        "strict_prompt_pass_at_8": pass8_metrics["strict_prompt_pass_at_8"],
        "strict_instruction_pass_at_8": pass8_metrics["strict_instruction_pass_at_8"],
        "per_sample_metrics": sample_metrics,
    }

    if RUN_GREEDY_PASS1 and all("greedy_response" in x for x in generations):
        print("=" * 80)
        print("Running official IFEval evaluator for greedy pass@1")
        greedy_response_path = write_ifeval_response_file(
            generations=generations,
            output_dir=output_dir,
            use_greedy=True,
        )
        greedy_strict_file = run_single_ifeval_official(
            input_path=input_path,
            response_path=greedy_response_path,
            official_dir=root_official_dir / "greedy",
        )
        greedy_records = read_ifeval_records(greedy_strict_file)
        greedy_metrics = metrics_from_ifeval_records(greedy_records)
        metrics["greedy_strict_prompt_accuracy"] = greedy_metrics["strict_prompt_accuracy"]
        metrics["greedy_strict_instruction_accuracy"] = greedy_metrics["strict_instruction_accuracy"]

    metrics_file = output_dir / f"ifeval_pass8_metrics_{len(generations)}_n{NUM_SAMPLES}.json"
    with open(metrics_file, "w", encoding="utf-8") as f:
        json.dump(metrics, f, ensure_ascii=False, indent=2)

    print(json.dumps(metrics, indent=2))
    print("Saved:", metrics_file)
    return metrics

In [9]:
all_summaries = []

for model_key in MODEL_KEYS_TO_RUN:
    model_info = MODEL_CONFIGS[model_key]
    model_name = model_info["model_name"]
    output_dir = BASE_OUTPUT_DIR / model_info["output_dir"]
    output_dir.mkdir(parents=True, exist_ok=True)

    print("\n" + "#" * 100)
    print("Evaluating:", model_info["display_name"])
    print("Model:", model_name)
    print("Output dir:", output_dir)
    print("RUN_CSQA_PASS8:", RUN_CSQA_PASS8)
    print("RUN_IFEVAL_PASS8:", RUN_IFEVAL_PASS8)
    print("NUM_SAMPLES:", NUM_SAMPLES)
    print("USE_4BIT:", USE_4BIT)
    print("#" * 100)

    tokenizer, model = load_deepseek_model(model_name)

    csqa_metrics = None
    if RUN_CSQA_PASS8:
        csqa_metrics = evaluate_commonsenseqa_pass8(
            tokenizer=tokenizer,
            model=model,
            model_info=model_info,
            output_dir=output_dir,
        )
    else:
        print("Skipping CommonsenseQA pass@8 because RUN_CSQA_PASS8 = False")

    ifeval_metrics = None
    generation_file = None
    if RUN_IFEVAL_PASS8:
        generation_file = generate_ifeval_pass8(
            tokenizer=tokenizer,
            model=model,
            model_info=model_info,
            output_dir=output_dir,
        )

        if RUN_IFEVAL_OFFICIAL:
            ifeval_metrics = run_ifeval_official_pass8(
                generation_file=generation_file,
                output_dir=output_dir,
            )
        else:
            print("Skipping official IFEval evaluator because RUN_IFEVAL_OFFICIAL = False")
    else:
        print("Skipping IFEval pass@8 because RUN_IFEVAL_PASS8 = False")

    summary = {
        "model": model_name,
        "display_name": model_info["display_name"],
        "stage": model_info["stage"],
        "num_samples": NUM_SAMPLES,
    }

    if csqa_metrics is not None:
        summary["commonsenseqa"] = {
            "split": "validation",
            "num_examples": csqa_metrics["num_examples"],
            "pass_at_1_sample0_accuracy": csqa_metrics["pass_at_1_sample0_accuracy"],
            "pass_at_8_accuracy": csqa_metrics["pass_at_8_accuracy"],
            "any_sample_parse_rate": csqa_metrics["any_sample_parse_rate"],
        }
        if "greedy_pass_at_1_accuracy" in csqa_metrics:
            summary["commonsenseqa"]["greedy_pass_at_1_accuracy"] = csqa_metrics["greedy_pass_at_1_accuracy"]

    if ifeval_metrics is not None:
        summary["ifeval_official"] = {
            "split": "train",
            "num_examples": ifeval_metrics["num_examples"],
            "sample0_strict_prompt_accuracy": ifeval_metrics["sample0_strict_prompt_accuracy"],
            "sample0_strict_instruction_accuracy": ifeval_metrics["sample0_strict_instruction_accuracy"],
            "strict_prompt_pass_at_8": ifeval_metrics["strict_prompt_pass_at_8"],
            "strict_instruction_pass_at_8": ifeval_metrics["strict_instruction_pass_at_8"],
        }
        if "greedy_strict_prompt_accuracy" in ifeval_metrics:
            summary["ifeval_official"]["greedy_strict_prompt_accuracy"] = ifeval_metrics["greedy_strict_prompt_accuracy"]
            summary["ifeval_official"]["greedy_strict_instruction_accuracy"] = ifeval_metrics["greedy_strict_instruction_accuracy"]
    elif generation_file is not None:
        summary["ifeval_generation_file"] = str(generation_file)

    with open(output_dir / "summary_pass8.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, ensure_ascii=False, indent=2)

    all_summaries.append(summary)

    print("\nSummary:")
    print(json.dumps(summary, indent=2))

    unload_model(tokenizer, model)

with open(BASE_OUTPUT_DIR / "all_summaries_pass8.json", "w", encoding="utf-8") as f:
    json.dump(all_summaries, f, ensure_ascii=False, indent=2)

print("\nSaved combined summary:", BASE_OUTPUT_DIR / "all_summaries_pass8.json")
print(json.dumps(all_summaries, indent=2))


####################################################################################################
Evaluating: DeepSeek-Math-7B-Instruct
Model: deepseek-ai/deepseek-math-7b-instruct
Output dir: deepseek_math_eval_results/deepseek_math_7b_instruct_results
RUN_CSQA_PASS8: True
RUN_IFEVAL_PASS8: False
NUM_SAMPLES: 8
USE_4BIT: True
####################################################################################################
Loading deepseek-ai/deepseek-math-7b-instruct ...


config.json:   0%|          | 0.00/594 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pytorch_model.bin.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Loading weights:   0%|          | 0/273 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/121 [00:00<?, ?B/s]

Loaded: deepseek-ai/deepseek-math-7b-instruct
CommonsenseQA pass@8: DeepSeek-Math-7B-Instruct | total=200 | samples=8
CommonsenseQA pass@8 progress: 1/200
CommonsenseQA pass@8 progress: 25/200
CommonsenseQA pass@8 progress: 50/200
CommonsenseQA pass@8 progress: 75/200
CommonsenseQA pass@8 progress: 100/200
CommonsenseQA pass@8 progress: 125/200
CommonsenseQA pass@8 progress: 150/200
CommonsenseQA pass@8 progress: 175/200
CommonsenseQA pass@8 progress: 200/200
{
  "model": "deepseek-ai/deepseek-math-7b-instruct",
  "display_name": "DeepSeek-Math-7B-Instruct",
  "stage": "SFT / Instruct",
  "benchmark": "CommonsenseQA",
  "split": "validation",
  "num_examples": 200,
  "num_samples": 8,
  "pass_at_1_sample0_accuracy": 0.3,
  "pass_at_8_accuracy": 0.74,
  "pass_at_1_sample0_parse_rate": 0.955,
  "any_sample_parse_rate": 1.0,
  "sampling_temperature": 0.7,
  "sampling_top_p": 0.95
}
Saved: deepseek_math_eval_results/deepseek_math_7b_instruct_results/commonsenseqa_pass8_predictions_200_n8.j

In [10]:
import pandas as pd

summary_path = BASE_OUTPUT_DIR / "all_summaries_pass8.json"
with open(summary_path, "r", encoding="utf-8") as f:
    summaries = json.load(f)

rows = []
for s in summaries:
    row = {
        "Model": s["display_name"],
        "Stage": s["stage"],
        "Num samples": s.get("num_samples", NUM_SAMPLES),
    }

    if "commonsenseqa" in s:
        row["CSQA n"] = s["commonsenseqa"]["num_examples"]
        row["CSQA sample0 pass@1"] = s["commonsenseqa"]["pass_at_1_sample0_accuracy"]
        row["CSQA pass@8"] = s["commonsenseqa"]["pass_at_8_accuracy"]
        if "greedy_pass_at_1_accuracy" in s["commonsenseqa"]:
            row["CSQA greedy pass@1"] = s["commonsenseqa"]["greedy_pass_at_1_accuracy"]

    if "ifeval_official" in s:
        row["IFEval n"] = s["ifeval_official"]["num_examples"]
        row["IFEval sample0 prompt pass@1"] = s["ifeval_official"]["sample0_strict_prompt_accuracy"]
        row["IFEval sample0 instr pass@1"] = s["ifeval_official"]["sample0_strict_instruction_accuracy"]
        row["IFEval prompt pass@8"] = s["ifeval_official"]["strict_prompt_pass_at_8"]
        row["IFEval instr pass@8"] = s["ifeval_official"]["strict_instruction_pass_at_8"]
        if "greedy_strict_prompt_accuracy" in s["ifeval_official"]:
            row["IFEval greedy prompt pass@1"] = s["ifeval_official"]["greedy_strict_prompt_accuracy"]
            row["IFEval greedy instr pass@1"] = s["ifeval_official"]["greedy_strict_instruction_accuracy"]

    rows.append(row)

df = pd.DataFrame(rows)
display(df)

if len(df) >= 2:
    delta = {"Model": "Δ RL - SFT", "Stage": "Transfer effect"}
    numeric_cols = [c for c in df.columns if c not in ["Model", "Stage"]]
    for c in numeric_cols:
        if pd.api.types.is_numeric_dtype(df[c]):
            delta[c] = df.loc[1, c] - df.loc[0, c]
    df_with_delta = pd.concat([df, pd.DataFrame([delta])], ignore_index=True)
    display(df_with_delta)
    df_with_delta.to_csv(BASE_OUTPUT_DIR / "deepseek_sft_vs_rl_pass8_comparison.csv", index=False)
    print("Saved:", BASE_OUTPUT_DIR / "deepseek_sft_vs_rl_pass8_comparison.csv")

,Model,Stage,Num samples,CSQA n,CSQA sample0 pass@1,CSQA pass@8
0,DeepSeek-Math-7B-Instruct,SFT / Instruct,8,200,0.3,0.74
